In [1]:
# ============================================================
# FOUR-STAGE RESNET: CIFAR-10 COMPETITION
#
# Train from scratch using attached competition data only.
# No pretrained weights, external datasets, or ensembles.
#
# References:
# ResNet: https://arxiv.org/abs/1512.03385
# Cutout: https://arxiv.org/abs/1708.04552
# OneCycle: https://arxiv.org/abs/1708.07120
#
# GPU augmentation; images stored as uint8 to save GPU memory.
# ============================================================

import os
import math
import time
import json
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

EPOCHS = 200                 # Smoke test. Then use 100 in a NEW run.
BATCH = 128
LR = 0.1
WD = 5e-4

VAL_SIZE = 5000            # Keep 5000 for architecture experiments.
                          # 0 = full-data run; final epoch selected.

WIDTHS = (42, 84, 168, 336)
BLOCKS = (2, 2, 2, 2)
SEED = 42

LABEL_SMOOTHING = 0.1
CUTOUT_SIZE = 16
TTA = True                # Fixed horizontal-flip averaging.
EVAL_BATCH = 128

MODE = "train"            # "train" or "predict"
DATA_DIRECTORY = ""       # Blank = automatic competition detection.

# Recover only from YOUR matching interrupted run.
RESUME_CHECKPOINT = ""

# For MODE="predict": path to this code's best.pt or final.pt.
PREDICT_CHECKPOINT = ""

assert MODE in {"train", "predict"}
assert EPOCHS >= 2
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings."

dev = torch.device("cuda:0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False

CONFIG = {
    "architecture": "four_stage_basic_resnet",
    "epochs": EPOCHS,
    "batch_size": BATCH,
    "learning_rate": LR,
    "weight_decay": WD,
    "validation_size": VAL_SIZE,
    "widths": list(WIDTHS),
    "blocks": list(BLOCKS),
    "seed": SEED,
    "label_smoothing": LABEL_SMOOTHING,
    "cutout_size": CUTOUT_SIZE,
    "horizontal_flip_averaging": TTA,
    "optimizer": "SGD",
    "momentum": 0.9,
    "nesterov": True,
    "scheduler": "OneCycleLR",
    "pct_start": 0.15,
    "pretrained": False,
}

# Every execution has a separate output folder.
run_name = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUT = Path("/kaggle/working") / f"four_stage_resnet_{run_name}"
OUT.mkdir(parents=True, exist_ok=False)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Output:", OUT)
print("One GPU is used.")

if MODE == "train" and EPOCHS == 5:
    print("SMOKE TEST: do not submit these five-epoch predictions.")


# ============================================================
# 2. FIND COMPETITION METADATA
# ============================================================

if DATA_DIRECTORY:
    ROOT = Path(DATA_DIRECTORY)
else:
    matches = [
        p.parent
        for p in Path("/kaggle/input").rglob("train_labels.csv")
        if (p.parent / "test.csv").is_file()
        and (p.parent / "sample_submission.csv").is_file()
    ]

    assert len(matches) == 1, (
        f"Set DATA_DIRECTORY explicitly. Candidates: {matches}"
    )
    ROOT = matches[0]

train_df = pd.read_csv(ROOT / "train_labels.csv")
test_df = pd.read_csv(ROOT / "test.csv")
sub_df = pd.read_csv(ROOT / "sample_submission.csv")

assert {"id", "filename", "label"} <= set(train_df.columns)
assert {"id", "filename"} <= set(test_df.columns)
assert sub_df.columns.tolist() == ["id", "label"]
assert len(train_df) == 50000

for frame in [train_df, test_df, sub_df]:
    assert frame.id.is_unique
    assert frame.id.notna().all()

assert train_df[["filename", "label"]].notna().all().all()
assert test_df.filename.notna().all()
assert train_df.filename.is_unique
assert test_df.filename.is_unique

assert train_df.label.between(0, 9).all()
assert (train_df.label == train_df.label.astype(np.int64)).all()

assert len(test_df) == len(sub_df)
assert set(test_df.id) == set(sub_df.id)

# Load evaluation images in sample-submission order.
ordered_test = sub_df[["id"]].merge(
    test_df[["id", "filename"]],
    on="id",
    how="left",
    sort=False,
    validate="one_to_one",
)

assert ordered_test.filename.notna().all()
assert ordered_test.id.equals(sub_df.id)

print("Data:", ROOT)
print(train_df.head(2))


# ============================================================
# 3. LOAD IMAGES BY THE EXPLICIT FILENAME COLUMN
# ============================================================

def image_directory(folder):
    direct = ROOT / folder
    if direct.is_dir():
        return direct

    candidates = [p for p in ROOT.rglob(folder) if p.is_dir()]
    assert len(candidates) == 1, f"Cannot locate {folder}"
    return candidates[0]


def load_images(frame, folder):
    base = image_directory(folder)
    images = np.empty((len(frame), 32, 32, 3), dtype=np.uint8)

    for index, filename in enumerate(frame.filename.astype(str)):
        path = base / filename
        if not path.is_file():
            path = ROOT / filename

        assert path.is_file(), f"Missing image: {path}"

        with Image.open(path) as image:
            array = np.array(image.convert("RGB"), copy=True)

        assert array.shape == (32, 32, 3), (path, array.shape)
        images[index] = array

    return images


print("Loading training images...")
Xtr = load_images(train_df, "train_images")
ytr = train_df.label.to_numpy(dtype=np.int64)

fingerprinter = hashlib.sha256()
fingerprinter.update(train_df.to_csv(index=False).encode())
fingerprinter.update(memoryview(Xtr))
training_fingerprint = fingerprinter.hexdigest()


# ============================================================
# 4. STRATIFIED SPLIT AND FITTING-ONLY NORMALIZATION
# ============================================================

all_indices = np.arange(len(ytr))

if VAL_SIZE:
    tr_indices, va_indices = train_test_split(
        all_indices,
        test_size=VAL_SIZE,
        stratify=ytr,
        random_state=SEED,
    )
else:
    tr_indices = all_indices
    va_indices = np.array([], dtype=np.int64)

assert len(np.intersect1d(tr_indices, va_indices)) == 0

np.savez(
    OUT / "data_split.npz",
    training=tr_indices,
    validation=va_indices,
)

fit_pixels = Xtr[tr_indices]

mean_np = fit_pixels.mean(
    axis=(0, 1, 2), dtype=np.float64
) / 255.0

std_np = fit_pixels.std(
    axis=(0, 1, 2), dtype=np.float64
) / 255.0

del fit_pixels

assert np.isfinite(mean_np).all()
assert np.isfinite(std_np).all() and (std_np > 0).all()

# Keep the complete training image array as uint8 on GPU.
# Normalize only individual batches.
Xtr_gpu = (
    torch.from_numpy(Xtr)
    .permute(0, 3, 1, 2)
    .contiguous()
    .to(dev)
)

ytr_gpu = torch.from_numpy(ytr).to(dev)

tr_gpu = torch.as_tensor(tr_indices, device=dev)
va_gpu = torch.as_tensor(va_indices, device=dev)

mean = torch.tensor(
    mean_np, device=dev, dtype=torch.float32
).view(1, 3, 1, 1)

std = torch.tensor(
    std_np, device=dev, dtype=torch.float32
).view(1, 3, 1, 1)

print("Training:", len(tr_indices))
print("Validation:", len(va_indices))
print("Mean:", mean_np)
print("Std:", std_np)


# ============================================================
# 5. GPU NORMALIZATION AND AUGMENTATION
# ============================================================

def normalize(batch):
    return (batch.float() / 255.0 - mean) / std


def augment(batch):
    # Input: normalized float tensor, B x 3 x 32 x 32.
    batch_size = batch.size(0)

    flip = torch.rand(batch_size, device=dev) < 0.5
    batch = torch.where(
        flip[:, None, None, None],
        batch.flip(-1),
        batch,
    )

    padded = F.pad(batch, (4, 4, 4, 4), mode="reflect")
    axis = torch.arange(32, device=dev)

    rows = (
        torch.randint(0, 9, (batch_size, 1), device=dev)
        + axis
    )
    columns = (
        torch.randint(0, 9, (batch_size, 1), device=dev)
        + axis
    )

    batch = padded[
        torch.arange(batch_size, device=dev)[:, None, None, None],
        torch.arange(3, device=dev)[None, :, None, None],
        rows[:, None, :, None],
        columns[:, None, None, :],
    ]

    # Exactly CUTOUT_SIZE pixels per dimension away from borders.
    cy = torch.randint(0, 32, (batch_size, 1, 1), device=dev)
    cx = torch.randint(0, 32, (batch_size, 1, 1), device=dev)

    yy = axis.view(1, 32, 1)
    xx = axis.view(1, 1, 32)
    half = CUTOUT_SIZE // 2

    mask = (
        (yy >= cy - half)
        & (yy < cy - half + CUTOUT_SIZE)
        & (xx >= cx - half)
        & (xx < cx - half + CUTOUT_SIZE)
    )

    return batch.masked_fill(mask.unsqueeze(1), 0)


# ============================================================
# 6. MODEL
# ============================================================

class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()

        self.c1 = nn.Conv2d(
            cin, cout, 3, stride, 1, bias=False
        )
        self.b1 = nn.BatchNorm2d(cout)

        self.c2 = nn.Conv2d(
            cout, cout, 3, 1, 1, bias=False
        )
        self.b2 = nn.BatchNorm2d(cout)

        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride, bias=False),
                nn.BatchNorm2d(cout),
            )

    def forward(self, x):
        residual = F.relu(self.b1(self.c1(x)))
        residual = self.b2(self.c2(residual))

        shortcut = x if self.sc is None else self.sc(x)
        return F.relu(residual + shortcut)


class ResNet(nn.Module):
    def __init__(self, widths, blocks, nc=10):
        super().__init__()

        assert len(widths) == len(blocks)

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, widths[0], 3, 1, 1, bias=False
            ),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(),
        )

        layers = []
        cin = widths[0]

        for stage, (width, count) in enumerate(zip(widths, blocks)):
            for block_index in range(count):
                stride = (
                    2 if stage > 0 and block_index == 0 else 1
                )
                layers.append(Block(cin, width, stride))
                cin = width

        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(cin, nc)

    def forward(self, x):
        x = self.layers(self.stem(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)


model = (
    ResNet(WIDTHS, BLOCKS)
    .to(dev)
    .to(memory_format=torch.channels_last)
)

n_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Trainable parameters: {n_params:,}")
assert n_params <= 5_000_000, "Over the 5M limit!"

model.eval()
with torch.inference_mode():
    assert model(
        torch.zeros(2, 3, 32, 32, device=dev)
    ).shape == (2, 10)


# ============================================================
# 7. PREDICTION AND CHECKPOINT HELPERS
# ============================================================

@torch.inference_mode()
def predict(images_uint8, indices=None, tta=True):
    model.eval()
    outputs = []

    count = (
        len(images_uint8)
        if indices is None
        else len(indices)
    )

    for start in range(0, count, EVAL_BATCH):
        if indices is None:
            pixels = images_uint8[start:start + EVAL_BATCH]
        else:
            pixels = images_uint8[
                indices[start:start + EVAL_BATCH]
            ]

        batch = normalize(pixels).contiguous(
            memory_format=torch.channels_last
        )

        with torch.autocast("cuda", dtype=torch.float16):
            probabilities = model(batch).float().softmax(1)

            if tta:
                flipped = model(batch.flip(-1)).float().softmax(1)
                probabilities = (probabilities + flipped) / 2

        outputs.append(probabilities.cpu())

    return torch.cat(outputs)


def cpu_state():
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def save_safely(payload, path):
    temporary = path.with_suffix(".temporary")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def inference_checkpoint(weights, epoch, accuracy):
    return {
        "model": weights,
        "config": CONFIG,
        "parameters": n_params,
        "epoch": epoch,
        "validation_accuracy": accuracy,
        "mean": mean_np.tolist(),
        "std": std_np.tolist(),
        "training_fingerprint": training_fingerprint,
        "training_indices": tr_indices,
        "validation_indices": va_indices,
    }


selected_path = OUT / ("best.pt" if VAL_SIZE else "final.pt")


# ============================================================
# 8. TRAINING
# ============================================================

if MODE == "train":
    decay, no_decay = [], []

    for parameter in model.parameters():
        if parameter.requires_grad:
            if parameter.ndim <= 1:
                no_decay.append(parameter)
            else:
                decay.append(parameter)

    opt = torch.optim.SGD(
        [
            {"params": decay, "weight_decay": WD},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=LR,
        momentum=0.9,
        nesterov=True,
    )

    # Include the final partial batch instead of discarding it.
    steps = math.ceil(len(tr_indices) / BATCH)

    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=LR,
        total_steps=EPOCHS * steps,
        pct_start=0.15,
    )

    scaler = torch.amp.GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    start_epoch = 0
    best_accuracy = -1.0
    best_epoch = 0
    best_weights = None
    history = []

    if RESUME_CHECKPOINT:
        # Load only your own trusted checkpoints.
        saved = torch.load(
            RESUME_CHECKPOINT,
            map_location="cpu",
            weights_only=False,
        )

        assert saved["config"] == CONFIG, (
            "Resume requires identical settings, including EPOCHS."
        )
        assert saved["training_fingerprint"] == training_fingerprint
        assert np.array_equal(saved["training_indices"], tr_indices)
        assert np.array_equal(saved["validation_indices"], va_indices)

        model.load_state_dict(saved["model"])
        opt.load_state_dict(saved["optimizer"])
        sched.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])

        start_epoch = saved["completed_epochs"]
        best_accuracy = saved["best_accuracy"]
        best_epoch = saved["best_epoch"]
        best_weights = saved["best_weights"]
        history = saved["history"]

        random.setstate(saved["python_rng"])
        np.random.set_state(saved["numpy_rng"])
        torch.set_rng_state(saved["torch_rng"])
        torch.cuda.set_rng_state_all(saved["cuda_rng"])

        if VAL_SIZE and best_weights is not None:
            save_safely(
                inference_checkpoint(
                    best_weights, best_epoch, best_accuracy
                ),
                selected_path,
            )

        print("Resuming at epoch:", start_epoch + 1)

    (OUT / "config.json").write_text(
        json.dumps(
            {
                **CONFIG,
                "parameters": n_params,
                "mean": mean_np.tolist(),
                "std": std_np.tolist(),
                "training_fingerprint": training_fingerprint,
                "torch_version": str(torch.__version__),
            },
            indent=2,
        )
    )

    for epoch in range(start_epoch, EPOCHS):
        started = time.time()
        model.train()

        permutation = tr_gpu[
            torch.randperm(len(tr_gpu), device=dev)
        ]

        total_loss = 0.0
        total_examples = 0

        for start in range(0, len(permutation), BATCH):
            batch_indices = permutation[start:start + BATCH]

            inputs = augment(
                normalize(Xtr_gpu[batch_indices])
            ).contiguous(memory_format=torch.channels_last)

            targets = ytr_gpu[batch_indices]

            opt.zero_grad(set_to_none=True)

            with torch.autocast("cuda", dtype=torch.float16):
                loss = criterion(model(inputs), targets)

            if not bool(torch.isfinite(loss)):
                raise RuntimeError("Nonfinite loss.")

            previous_scale = scaler.get_scale()

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            # Avoid advancing the schedule when AMP skips an update.
            if scaler.get_scale() >= previous_scale:
                sched.step()

            total_loss += loss.item() * len(targets)
            total_examples += len(targets)

        validation_accuracy = None

        if VAL_SIZE:
            probabilities = predict(
                Xtr_gpu,
                indices=va_gpu,
                tta=TTA,
            )

            validation_accuracy = (
                probabilities.argmax(1).numpy()
                == ytr[va_indices]
            ).mean().item()

            if validation_accuracy > best_accuracy:
                best_accuracy = validation_accuracy
                best_epoch = epoch + 1
                best_weights = cpu_state()

                save_safely(
                    inference_checkpoint(
                        best_weights,
                        best_epoch,
                        best_accuracy,
                    ),
                    selected_path,
                )

        seconds = time.time() - started

        history.append({
            "epoch": epoch + 1,
            "training_loss": total_loss / total_examples,
            "validation_accuracy": validation_accuracy,
            "lr_next_step": opt.param_groups[0]["lr"],
            "seconds": seconds,
        })

        pd.DataFrame(history).to_csv(
            OUT / "training_history.csv",
            index=False,
        )

        save_safely(
            {
                "model": model.state_dict(),
                "optimizer": opt.state_dict(),
                "scheduler": sched.state_dict(),
                "scaler": scaler.state_dict(),
                "completed_epochs": epoch + 1,
                "best_accuracy": best_accuracy,
                "best_epoch": best_epoch,
                "best_weights": best_weights,
                "history": history,
                "config": CONFIG,
                "training_fingerprint": training_fingerprint,
                "training_indices": tr_indices,
                "validation_indices": va_indices,
                "python_rng": random.getstate(),
                "numpy_rng": np.random.get_state(),
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all(),
            },
            OUT / "last.pt",
        )

        remaining = seconds * (EPOCHS - epoch - 1) / 3600

        message = (
            f"Epoch {epoch + 1:03d}/{EPOCHS} | "
            f"Loss {total_loss / total_examples:.4f} | "
            f"{seconds:.0f}s | Remaining ~{remaining:.2f}h"
        )

        if VAL_SIZE:
            message += (
                f" | Val {validation_accuracy:.2%}"
                f" | Best {best_accuracy:.2%}"
            )

        print(message, flush=True)

    if not VAL_SIZE:
        # No validation-based selection in a full-data run.
        save_safely(
            inference_checkpoint(cpu_state(), EPOCHS, None),
            selected_path,
        )

else:
    assert PREDICT_CHECKPOINT, (
        "Set PREDICT_CHECKPOINT to your saved best.pt or final.pt."
    )
    selected_path = Path(PREDICT_CHECKPOINT)


# ============================================================
# 9. LOAD SELECTED MODEL AND VERIFY VALIDATION
# ============================================================

checkpoint = torch.load(
    selected_path,
    map_location="cpu",
    weights_only=False,
)

for key in ["architecture", "widths", "blocks"]:
    assert checkpoint["config"][key] == CONFIG[key], (
        f"Checkpoint mismatch: {key}"
    )

assert checkpoint["training_fingerprint"] == training_fingerprint

model.load_state_dict(checkpoint["model"])
model.eval()

mean_np = np.asarray(checkpoint["mean"])
std_np = np.asarray(checkpoint["std"])

mean = torch.tensor(
    mean_np, device=dev, dtype=torch.float32
).view(1, 3, 1, 1)

std = torch.tensor(
    std_np, device=dev, dtype=torch.float32
).view(1, 3, 1, 1)

use_tta = checkpoint["config"]["horizontal_flip_averaging"]
saved_validation = np.asarray(checkpoint["validation_indices"])

verified_accuracy = None

if len(saved_validation):
    validation_probabilities = predict(
        Xtr_gpu,
        indices=torch.as_tensor(saved_validation, device=dev),
        tta=use_tta,
    )

    validation_predictions = (
        validation_probabilities.argmax(1).numpy()
    )

    verified_accuracy = float(
        (validation_predictions == ytr[saved_validation]).mean()
    )

    pd.DataFrame({
        "id": train_df.iloc[saved_validation].id.to_numpy(),
        "true_label": ytr[saved_validation],
        "prediction": validation_predictions,
    }).to_csv(
        OUT / "validation_predictions.csv",
        index=False,
    )

    print(
        f"Selected epoch: {checkpoint['epoch']} | "
        f"Verified validation accuracy: {verified_accuracy:.2%}"
    )


# ============================================================
# 10. GENERATE SUBMISSION IN SAMPLE ORDER
# ============================================================

print("Loading evaluation images for inference...")
Xte = load_images(ordered_test, "test_images")

Xte_gpu = (
    torch.from_numpy(Xte)
    .permute(0, 3, 1, 2)
    .contiguous()
    .to(dev)
)

test_probabilities = predict(Xte_gpu, tta=use_tta)
predictions = test_probabilities.argmax(1).numpy().astype(np.int64)

submission = sub_df[["id"]].copy()
submission["label"] = predictions

assert submission.columns.tolist() == ["id", "label"]
assert len(submission) == len(test_df)
assert submission.id.equals(sub_df.id)
assert submission.id.is_unique
assert submission.notna().all().all()
assert submission.label.between(0, 9).all()

submission_path = OUT / "submission.csv"
submission.to_csv(submission_path, index=False)

written = pd.read_csv(submission_path)

assert written.id.equals(sub_df.id)
assert np.array_equal(written.label.to_numpy(), predictions)

(OUT / "results.json").write_text(
    json.dumps(
        {
            "parameters": n_params,
            "selected_epoch": checkpoint["epoch"],
            "validation_accuracy": verified_accuracy,
            "training_images": len(checkpoint["training_indices"]),
            "submission_rows": len(submission),
            "horizontal_flip_averaging": use_tta,
            "smoke_test": checkpoint["config"]["epochs"] == 5,
            "submission_sha256": hashlib.sha256(
                submission_path.read_bytes()
            ).hexdigest(),
        },
        indent=2,
    )
)

print("\nCOMPLETE")
print("Selected checkpoint:", selected_path)
print("Submission:", submission_path)
print(submission.head())

if checkpoint["config"]["epochs"] == 5:
    print("\nSMOKE TEST ONLY: do not submit this CSV.")

# Download link for the current notebook session.
from IPython.display import display, FileLink

display(FileLink(str(submission_path)))

GPU: Tesla T4
PyTorch: 2.11.0+cu128
Output: /kaggle/working/four_stage_resnet_20261031_235638_805085
One GPU is used.
Data: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
   id         filename  label
0   0  train_00000.png      6
1   1  train_00001.png      9
Loading training images...
Training: 45000
Validation: 5000
Mean: [0.49113735 0.48211119 0.44644899]
Std: [0.24686914 0.24343013 0.26160339]
Trainable parameters: 4,815,940
Epoch 001/200 | Loss 1.6910 | 19s | Remaining ~1.03h | Val 51.88% | Best 51.88%
Epoch 002/200 | Loss 1.3982 | 16s | Remaining ~0.90h | Val 66.98% | Best 66.98%
Epoch 003/200 | Loss 1.2814 | 17s | Remaining ~0.91h | Val 71.60% | Best 71.60%
Epoch 004/200 | Loss 1.2037 | 17s | Remaining ~0.91h | Val 75.58% | Best 75.58%
Epoch 005/200 | Loss 1.1568 | 17s | Remaining ~0.92h | Val 78.06% | Best 78.06%
Epoch 006/200 | Loss 1.1123 | 17s | Remaining ~0.93h | Val 74.74% | Best 78.06%
Epoch 007/200 | Loss 1.0795 | 17s | Remaining ~0.94h | Val 79.2

/kaggle/working/four_stage_resnet_20261031_235638_805085/submission.csv